In [1]:
import subprocess
import os
import pandas as pd
import numpy as np
from core.reader import read_ansys_csv, load_experimental_data

ANSYS_EXE_PATH = r"D:\Program Files\ANSYS Inc\ANSYS Student\v252\ansys\bin\winx64\MAPDL.exe" 
WORKING_DIR = os.getcwd()

def run_ansys_simulation(params):
    with open('chab_params.txt', 'w') as f:
        for p in params:
            f.write(f"{p}\n")

    input_file = "chab.mac"
    output_file = "ansys.out"
    
    cmd = [
        ANSYS_EXE_PATH, 
        "-b",
        "-j", "opt_run",
        "-dir", WORKING_DIR, 
        "-i", input_file, 
        "-o", output_file
    ]

    try:
        subprocess.run(cmd, check=True, capture_output=True)
    except subprocess.CalledProcessError as e:
        print("Ошибка ANSYS:", e)
        return None

    try:
        df_res = read_ansys_csv("chab.csv")
        return df_res
    except Exception as e:
        print(f"Ошибка чтения CSV: {e}")
        return None

In [2]:
real_experiment_data_folder = "."
df_exp = load_experimental_data(real_experiment_data_folder)

zero_row = pd.DataFrame(0.0, columns=df_exp.columns, index=[0])
zero_row['Time'] = 1.0
df_exp = pd.concat([zero_row, df_exp]).reset_index(drop=True)

def objective_function(params):
    """
    Считает ошибку между экспериментом и моделью Шабоша.
    params: [sig_y, c1, g1, c2, g2, c3, g3]
    """
    print(f"Simulating: {params}")
    
    df_ansys = run_ansys_simulation(params)
    
    if df_ansys is None or len(df_ansys) != len(df_exp):
        return 1e9

    mse_zz = np.mean((df_ansys['S_ZZ'] - df_exp['S_ZZ'])**2)
    mse_tt = np.mean((df_ansys['S_TT'] - df_exp['S_TT'])**2)
    mse_tz = np.mean((df_ansys['S_TZ'] - df_exp['S_TZ'])**2)
    
    total_error = mse_zz + mse_tt + mse_tz
    print(f"Error: {total_error:.2f}")
    return total_error

In [3]:
import psutil

def kill_ansys_processes():
    for proc in psutil.process_iter():
        if proc.name() in ['ANSYS.exe', 'MAPDL.exe', 'ansys.exe']:
            proc.kill()

kill_ansys_processes()

In [4]:
from scipy.optimize import differential_evolution

# Границы поиска для каждого параметра
# [sig_y, c1, g1, c2, g2, c3, g3]
bounds = [
    (200, 400),      # Sig_Y
    (1e4, 5e5),      # C1 (Жесткая кинематика)
    (100, 5000),     # gamma1 (Быстрое насыщение)
    (1e3, 5e4),      # C2
    (10, 500),       # gamma2
    (100, 1e4),      # C3
    (0, 100)         # gamma3 (Линейная часть)
]

result = differential_evolution(
    objective_function, 
    bounds, 
    strategy='best1bin', 
    maxiter=15,      # Количество поколений (увеличьте до 20-50 для точности)
    popsize=10,       # Размер популяции (увеличьте до 10-15)
    disp=True,
    polish=True,
    workers=1
)

print("Оптимальные параметры найдены:")
print(result.x)

Simulating: [3.73383854e+02 2.17754654e+05 1.48358491e+03 5.26882105e+03
 1.83381420e+02 1.86559063e+03 2.60154796e+01]
Error: 29164.78
Simulating: [3.26390578e+02 2.46136609e+05 4.28009324e+03 4.60999129e+04
 3.27746250e+02 5.90001109e+02 2.23452874e+00]
Error: 13167.33
Simulating: [2.27447698e+02 4.48007141e+05 4.20772220e+03 3.39127031e+04
 2.48744170e+02 8.19618422e+03 1.17749019e+01]
Error: 13863.52
Simulating: [2.33783751e+02 1.76329781e+05 5.06031512e+02 3.93402896e+04
 3.57242466e+02 8.39229771e+03 9.66793100e+01]
Error: 111605.80
Simulating: [3.03405639e+02 8.47402410e+04 2.56279444e+03 4.90579858e+04
 1.24804118e+02 7.98695001e+03 8.12284863e+01]
Error: 66048.43
Simulating: [2.25683959e+02 4.17641153e+05 5.94289345e+02 1.34869075e+04
 2.37763612e+02 2.71112324e+03 2.47156415e+01]
Error: 389649.25
Simulating: [3.17963361e+02 2.98116206e+05 1.39250687e+03 2.10006507e+04
 8.83500719e+01 7.37039815e+03 4.56961780e+01]
Error: 118661.56
Simulating: [2.79905928e+02 4.74614237e+05 3.